# 34 SQL questions on Sakila

Each question is answered with one query, schema-qualified (`sakila.film`, not `film`). Comments explain choices where the question is ambiguous.

In [1]:
import os
import pandas as pd
from google.cloud.sql.connector import Connector

# Connect to Cloud SQL; credentials come from environment variables
connector = Connector()
cnx = connector.connect(
    os.environ["INSTANCE_CONNECTION_NAME"],
    "pymysql",
    user="root",
    password=os.environ["MYSQL_ROOT_PW"],
    db="sakila",
)
cursor = cnx.cursor()

# Helper: run one query and show the rows as a table
def run(sql):
    cursor.execute(sql)
    cols = [d[0] for d in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=cols)

run("SELECT DATABASE() AS db, VERSION() AS mysql_version")

,db,mysql_version
0,sakila,8.0.46-google


List all the movies (title & description) that are rated PG-13.

In [2]:
# Q1
run("""
SELECT title, description
FROM sakila.film
WHERE rating = 'PG-13'
""")

,title,description
0,AIRPLANE SIERRA,A Touching Saga of a Hunter And a Butler who m...
1,ALABAMA DEVIL,A Thoughtful Panorama of a Database Administra...
2,ALTER VICTORY,A Thoughtful Drama of a Composer And a Feminis...
3,ANTHEM LUKE,A Touching Panorama of a Waitress And a Woman ...
4,APOLLO TEEN,A Action-Packed Reflection of a Crocodile And ...
...,...,...
218,WHALE BIKINI,A Intrepid Story of a Pastry Chef And a Databa...
219,WHISPERER GIANT,A Intrepid Story of a Dentist And a Hunter who...
220,WORLD LEATHERNECKS,A Unbelieveable Tale of a Pioneer And a Astron...
221,WRONG BEHAVIOR,A Emotional Saga of a Crocodile And a Sumo Wre...


List all movies that are either PG OR PG-13 using the IN operator.

In [3]:
# Q2
run("""
SELECT title, rating
FROM sakila.film
WHERE rating IN ('PG', 'PG-13')
""")

,title,rating
0,ACADEMY DINOSAUR,PG
1,AGENT TRUMAN,PG
2,AIRPLANE SIERRA,PG-13
3,ALABAMA DEVIL,PG-13
4,ALASKA PHANTOM,PG
...,...,...
412,WORDS HUNTER,PG
413,WORLD LEATHERNECKS,PG-13
414,WORST BANGER,PG
415,WRONG BEHAVIOR,PG-13


List all staff members (first name, last name, email) whose password is NULL.

In [4]:
# Q3
run("""
SELECT first_name, last_name, email
FROM sakila.staff
WHERE password IS NULL
""")

,first_name,last_name,email
0,Jon,Stephens,Jon.Stephens@sakilastaff.com


Select all films that have title names like ZOO and rental duration greater than or equal to 4.

In [5]:
# Q4
run("""
SELECT title, rental_duration
FROM sakila.film
WHERE title LIKE '%ZOO%'
  AND rental_duration >= 4
""")

,title,rental_duration
0,MEMENTO ZOOLANDER,4
1,UNFORGIVEN ZOOLANDER,7
2,ZOOLANDER FICTION,5


What is the cost of renting the movie ACADEMY DINOSAUR for 2 weeks? Note: use a column alias and watch for the rental_duration value.

In [6]:
# Q5
run("""
SELECT ROUND(rental_rate * 14 / rental_duration, 2) AS cost_for_2_weeks
FROM sakila.film
WHERE title = 'ACADEMY DINOSAUR'
""")

,cost_for_2_weeks
0,2.31


**Comment (Q5):** `rental_rate` is the price for `rental_duration` days, and this film's `rental_duration` is 6, not 7. Two weeks is 14 days, so the cost is `rental_rate * 14 / rental_duration` = 0.99 x 14 / 6 = 2.31. (If a partial period had to be billed in full, it would be 3 periods = 2.97.)

List all unique districts where customers, staff, and stores are located. Note: check for NOT NULL values.

In [7]:
# Q6
run("""
SELECT a.district
FROM sakila.customer c
JOIN sakila.address a ON a.address_id = c.address_id
WHERE a.district IS NOT NULL AND a.district <> ''
UNION
SELECT a.district
FROM sakila.staff s
JOIN sakila.address a ON a.address_id = s.address_id
WHERE a.district IS NOT NULL AND a.district <> ''
UNION
SELECT a.district
FROM sakila.store st
JOIN sakila.address a ON a.address_id = st.address_id
WHERE a.district IS NOT NULL AND a.district <> ''
ORDER BY district
""")

,district
0,Abu Dhabi
1,Aceh
2,Adana
3,Addis Abeba
4,Aden
...,...
372,Zanzibar West
373,Zhejiang
374,Ziguinchor
375,Zufar


**Comment (Q6):** `UNION` (not `UNION ALL`) removes duplicates across the three sources. No district is NULL, but three customer addresses have a blank district (`''`), so those are excluded as well.

List the top 10 newest customers across all stores based on customer_id.

In [8]:
# Q7
run("""
SELECT customer_id, first_name, last_name, store_id
FROM sakila.customer
ORDER BY customer_id DESC
LIMIT 10
""")

,customer_id,first_name,last_name,store_id
0,599,AUSTIN,CINTRON,2
1,598,WADE,DELVALLE,1
2,597,FREDDIE,DUGGAN,1
3,596,ENRIQUE,FORSYTHE,1
4,595,TERRENCE,GUNDERSON,1
5,594,EDUARDO,HIATT,1
6,593,RENE,MCALISTER,2
7,592,TERRANCE,ROUSH,1
8,591,KENT,ARSENAULT,1
9,590,SETH,HANNON,2


Show the total number of movies.

In [9]:
# Q8
run("""
SELECT COUNT(*)
FROM sakila.film
""")

,COUNT(*)
0,1000


What is the minimum payment received and the max payment received across all transactions?

In [10]:
# Q9
run("""
SELECT MIN(amount), MAX(amount)
FROM sakila.payment
""")

,MIN(amount),MAX(amount)
0,0.00,11.99


Number of customers who rented movies between 02-2005 and 05-2005 (based on payment_date).

In [11]:
# Q10
run("""
SELECT COUNT(DISTINCT customer_id)
FROM sakila.payment
WHERE payment_date >= '2005-02-01'
  AND payment_date <  '2005-06-01'
""")

,COUNT(DISTINCT customer_id)
0,520


**Comment (Q10):** The window runs from 2005-02-01 up to, but not including, 2005-06-01, so all of May 2005 is counted. The earliest payment in Sakila is 2005-05-24, so every match comes from late May 2005.

List all movies where replacement_cost is greater than $15 or rental_duration is between 6 and 10 days.

In [12]:
# Q11
run("""
SELECT title, replacement_cost, rental_duration
FROM sakila.film
WHERE replacement_cost > 15
   OR rental_duration BETWEEN 6 AND 10
""")

,title,replacement_cost,rental_duration
0,ACADEMY DINOSAUR,20.99,6
1,ADAPTATION HOLES,18.99,7
2,AFFAIR PREJUDICE,26.99,5
3,AFRICAN EGG,22.99,6
4,AGENT TRUMAN,17.99,3
...,...,...,...
814,WYOMING STORM,29.99,6
815,YOUNG LANGUAGE,9.99,6
816,ZHIVAGO CORE,10.99,6
817,ZOOLANDER FICTION,28.99,5


What is the total amount spent by customers on movies in the year 2005?

In [13]:
# Q12
run("""
SELECT SUM(amount)
FROM sakila.payment
WHERE payment_date >= '2005-01-01'
  AND payment_date <  '2006-01-01'
""")

,SUM(amount)
0,66892.38


**Comment (Q12):** Payments dated February 2006 (182 rows) fall outside 2005 and are excluded.

What is the average replacement cost across all movies?

In [14]:
# Q13
run("""
SELECT AVG(replacement_cost)
FROM sakila.film
""")

,AVG(replacement_cost)
0,19.984000


What is the standard deviation of rental_rate across all movies?

In [15]:
# Q14
run("""
SELECT STDDEV_POP(rental_rate)
FROM sakila.film
""")

,STDDEV_POP(rental_rate)
0,1.64557


**Comment (Q14):** `STDDEV_POP` is the population standard deviation (the same as MySQL's `STDDEV`), because the question covers all movies. `STDDEV_SAMP` would give the sample version.

What is the midrange of rental_duration for all movies? ((MAX + MIN) / 2)

In [16]:
# Q15
run("""
SELECT (MAX(rental_duration) + MIN(rental_duration)) / 2
FROM sakila.film
""")

,(MAX(rental_duration) + MIN(rental_duration)) / 2
0,5.0000


Customers sorted by first name and last name in ascending order.

In [17]:
# Q16
run("""
SELECT customer_id, first_name, last_name
FROM sakila.customer
ORDER BY first_name ASC, last_name ASC
""")

,customer_id,first_name,last_name
0,375,AARON,SELBY
1,367,ADAM,GOOCH
2,525,ADRIAN,CLARY
3,217,AGNES,BISHOP
4,389,ALAN,KAHN
...,...,...,...
594,359,WILLIE,MARKHAM
595,212,WILMA,RICHARDS
596,190,YOLANDA,WEAVER
597,174,YVONNE,WATKINS


Count of movies that are either G / NC-17 / PG-13 / PG / R, grouped by rating.

In [18]:
# Q17
run("""
SELECT rating, COUNT(*)
FROM sakila.film
WHERE rating IN ('G', 'NC-17', 'PG-13', 'PG', 'R')
GROUP BY rating
""")

,rating,COUNT(*)
0,PG,194
1,G,178
2,NC-17,210
3,PG-13,223
4,R,195


Number of addresses in each district.

In [19]:
# Q18
run("""
SELECT district, COUNT(*)
FROM sakila.address
GROUP BY district
ORDER BY district
""")

,district,COUNT(*)
0,,3
1,Abu Dhabi,2
2,Aceh,1
3,Adana,2
4,Addis Abeba,1
...,...,...
373,Zanzibar West,1
374,Zhejiang,2
375,Ziguinchor,1
376,Zufar,1


Find the movies where the rental_rate is greater than $1, ordered descending.

In [20]:
# Q19
run("""
SELECT title, rental_rate
FROM sakila.film
WHERE rental_rate > 1
ORDER BY rental_rate DESC
""")

,title,rental_rate
0,ACE GOLDFINGER,4.99
1,AIRPLANE SIERRA,4.99
2,AIRPORT POLLOCK,4.99
3,ALADDIN CALENDAR,4.99
4,ALI FOREVER,4.99
...,...,...
654,WORDS HUNTER,2.99
655,WORKER TARZAN,2.99
656,WORST BANGER,2.99
657,WRONG BEHAVIOR,2.99


Top 2 movies rated R with the highest replacement_cost.

In [21]:
# Q20
run("""
SELECT title, rating, replacement_cost
FROM sakila.film
WHERE rating = 'R'
ORDER BY replacement_cost DESC, title
LIMIT 2
""")

,title,rating,replacement_cost
0,CHARIOTS CONSPIRACY,R,29.99
1,CUPBOARD SINNERS,R,29.99


**Comment (Q20):** Ten R-rated films tie at the highest replacement cost (29.99), so `title` breaks the tie and the two rows are repeatable.

Find the most frequently occurring (mode) rental_rate across products.

In [22]:
# Q21
run("""
SELECT rental_rate, COUNT(*)
FROM sakila.film
GROUP BY rental_rate
ORDER BY COUNT(*) DESC
LIMIT 1
""")

,rental_rate,COUNT(*)
0,0.99,341


Find the 2 longest movies with length greater than 50 minutes that have commentaries as a special feature.

In [23]:
# Q22
run("""
SELECT title, length, special_features
FROM sakila.film
WHERE length > 50
  AND special_features LIKE '%Commentaries%'
ORDER BY length DESC, title
LIMIT 2
""")

,title,length,special_features
0,CONTROL ANTHEM,185,Commentaries
1,HOME PITY,185,"Trailers,Commentaries,Behind the Scenes"


**Comment (Q22):** Four films with commentaries tie at the longest length (185 minutes), so `title` breaks the tie and the two rows are repeatable.

List the years that have more than 2 movies released.

In [24]:
# Q23
run("""
SELECT release_year
FROM sakila.film
GROUP BY release_year
HAVING COUNT(*) > 2
""")

,release_year
0,2006


List the actors (first_name, last_name) who acted in more than 25 movies. Note: also show the count of movies per actor.

In [25]:
# Q24
run("""
SELECT a.first_name, a.last_name, COUNT(fa.film_id)
FROM sakila.actor a
JOIN sakila.film_actor fa ON fa.actor_id = a.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
HAVING COUNT(fa.film_id) > 25
""")

,first_name,last_name,COUNT(fa.film_id)
0,JOHNNY,LOLLOBRIGIDA,29
1,GRACE,MOSTEL,30
2,KARL,BERRY,31
3,UMA,WOOD,35
4,VIVIEN,BERGEN,30
...,...,...,...
120,JOHN,SUVARI,29
121,JAYNE,SILVERSTONE,27
122,BELA,WALKEN,30
123,REESE,WEST,33


**Comment (Q24):** Grouped by `actor_id` as well as the names, because two different actors share the name SUSAN DAVIS.

List the actors who have worked in German-language movies.

In [26]:
# Q25
run("""
SELECT DISTINCT a.actor_id, a.first_name, a.last_name
FROM sakila.actor a
JOIN sakila.film_actor fa ON fa.actor_id = a.actor_id
JOIN sakila.film f ON f.film_id = fa.film_id
JOIN sakila.language l ON l.language_id = f.language_id
WHERE l.name = 'German'
""")

,actor_id,first_name,last_name


**Comment (Q25):** The result is empty. German exists in `sakila.language`, but all 1,000 films have `language_id = 1` (English) and `original_language_id` is NULL for every film, so no actor has worked in a German-language movie in this data.

List all customers who rented more than 3 horror movies.

In [27]:
# Q26
run("""
SELECT c.customer_id, c.first_name, c.last_name
FROM sakila.customer c
JOIN sakila.rental r ON r.customer_id = c.customer_id
JOIN sakila.inventory i ON i.inventory_id = r.inventory_id
JOIN sakila.film_category fc ON fc.film_id = i.film_id
JOIN sakila.category cat ON cat.category_id = fc.category_id
WHERE cat.name = 'Horror'
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(*) > 3
""")

,customer_id,first_name,last_name
0,268,NINA,SOTO
1,197,SUE,PETERS
2,148,ELEANOR,HUNT
3,459,TOMMY,COLLAZO
4,449,OSCAR,AQUINO
5,72,THERESA,WATSON
6,332,STEPHEN,QUALLS
7,513,DUANE,TUBBS
8,305,RICHARD,MCCRARY
9,507,EDGAR,RHOADS


**Comment (Q26):** Counts rentals of Horror films, so renting the same film twice counts twice. Counting distinct films instead (`HAVING COUNT(DISTINCT i.film_id) > 3`) would drop two customers (34 to 32).

List all customers who rented a movie starring SCARLETT BENING.

In [28]:
# Q27
run("""
SELECT DISTINCT c.customer_id, c.first_name, c.last_name
FROM sakila.customer c
JOIN sakila.rental r ON r.customer_id = c.customer_id
JOIN sakila.inventory i ON i.inventory_id = r.inventory_id
JOIN sakila.film_actor fa ON fa.film_id = i.film_id
JOIN sakila.actor a ON a.actor_id = fa.actor_id
WHERE a.first_name = 'SCARLETT'
  AND a.last_name = 'BENING'
""")

,customer_id,first_name,last_name
0,59,CHERYL,MURPHY
1,339,WALTER,PERRYMAN
2,1,MARY,SMITH
3,128,MARJORIE,TUCKER
4,29,ANGELA,HERNANDEZ
...,...,...,...
257,324,GARY,COY
258,483,VERNON,CHAPA
259,522,ARNOLD,HAVENS
260,320,ANTHONY,SCHWAB


Customers residing at postal code 62703 who rented movies that were Documentaries.

In [29]:
# Q28
run("""
SELECT DISTINCT c.customer_id, c.first_name, c.last_name
FROM sakila.customer c
JOIN sakila.address a ON a.address_id = c.address_id
JOIN sakila.rental r ON r.customer_id = c.customer_id
JOIN sakila.inventory i ON i.inventory_id = r.inventory_id
JOIN sakila.film_category fc ON fc.film_id = i.film_id
JOIN sakila.category cat ON cat.category_id = fc.category_id
WHERE a.postal_code = '62703'
  AND cat.name = 'Documentary'
""")

,customer_id,first_name,last_name
0,582,ANDY,VANHORN


Find all addresses where address2 is not empty (contains some text), returned sorted.

In [30]:
# Q29
run("""
SELECT address_id, address, address2, district, city_id, postal_code, phone
FROM sakila.address
WHERE address2 IS NOT NULL
  AND address2 <> ''
ORDER BY address2
""")

,address_id,address,address2,district,city_id,postal_code,phone


**Comment (Q29):** The result is empty. In this data `address2` is NULL for 4 addresses and an empty string (`''`) for the other 599, so none contains text. Testing only `IS NOT NULL` would wrongly return those 599 blank rows.

How many films involve a "Crocodile" and a "Shark" based on the film description?

In [31]:
# Q30
run("""
SELECT COUNT(*)
FROM sakila.film
WHERE description LIKE '%Crocodile%'
  AND description LIKE '%Shark%'
""")

,COUNT(*)
0,10


List the actors who played in a film involving a "Crocodile" and a "Shark", with the release year, sorted by the actors' last names.

In [32]:
# Q31
run("""
SELECT DISTINCT a.first_name, a.last_name, f.release_year
FROM sakila.actor a
JOIN sakila.film_actor fa ON fa.actor_id = a.actor_id
JOIN sakila.film f ON f.film_id = fa.film_id
WHERE f.description LIKE '%Crocodile%'
  AND f.description LIKE '%Shark%'
ORDER BY a.last_name, a.first_name
""")

,first_name,last_name,release_year
0,KIRSTEN,AKROYD,2006
1,KIM,ALLEN,2006
2,AUDREY,BAILEY,2006
3,JULIA,BARRYMORE,2006
4,VIVIEN,BASINGER,2006
5,VIVIEN,BERGEN,2006
6,HENRY,BERRY,2006
7,KARL,BERRY,2006
8,LAURA,BRODY,2006
9,JOHNNY,CAGE,2006


Find all film categories with between 55 and 65 films. Return category name and film count, sorted highest to lowest.

In [33]:
# Q32
run("""
SELECT c.name, COUNT(*)
FROM sakila.category c
JOIN sakila.film_category fc ON fc.category_id = c.category_id
GROUP BY c.category_id, c.name
HAVING COUNT(*) BETWEEN 55 AND 65
ORDER BY COUNT(*) DESC
""")

,name,COUNT(*)
0,Action,64
1,New,63
2,Drama,62
3,Games,61
4,Sci-Fi,61
5,Children,60
6,Comedy,58
7,Classics,57
8,Travel,57
9,Horror,56


In which film categories is the average difference between replacement_cost and rental_rate greater than $17?

In [34]:
# Q33
run("""
SELECT c.name
FROM sakila.category c
JOIN sakila.film_category fc ON fc.category_id = c.category_id
JOIN sakila.film f ON f.film_id = fc.film_id
GROUP BY c.category_id, c.name
HAVING AVG(f.replacement_cost - f.rental_rate) > 17
""")

,name
0,Action
1,Animation
2,Children
3,Classics
4,Drama
5,Games
6,Sci-Fi
7,Sports


Overdue-rentals report: search rental for rows with return_date IS NULL where rental_date is further in the past than the film's rental_duration. Return the film name, customer name, and phone number.

In [35]:
# Q34
run("""
SELECT f.title, c.first_name, c.last_name, a.phone
FROM sakila.rental r
JOIN sakila.inventory i ON i.inventory_id = r.inventory_id
JOIN sakila.film f ON f.film_id = i.film_id
JOIN sakila.customer c ON c.customer_id = r.customer_id
JOIN sakila.address a ON a.address_id = c.address_id
WHERE r.return_date IS NULL
  AND DATEDIFF(CURRENT_DATE(), r.rental_date) > f.rental_duration
""")

,title,first_name,last_name,phone
0,HYDE DOCTOR,GAIL,KNIGHT,904253967161
1,HUNGER ROOF,GREGORY,MAULDIN,80303246192
2,FRISCO FORREST,LOUISE,JENKINS,800716535041
3,TITANS JERK,WILLIE,HOWELL,991802825778
4,CONNECTION MICROCOSMOS,EMILY,DIAZ,333339908719
...,...,...,...,...
178,DANCES NONE,JESSIE,BANKS,352679173732
179,CONVERSATION DOWNHILL,RAFAEL,ABNEY,82671830126
180,SHOCK CABIN,STEPHANIE,MITCHELL,42384721397
181,WEDDING APOLLO,REGINA,BERRY,201705577290


**Comment (Q34):** All of Sakila's rentals are dated 2005-2006, so measured against today every rental with a NULL `return_date` is overdue. The customer name is returned as separate `first_name` and `last_name` columns.

# Checking an AI assistant against MySQL

Claude explains; MySQL decides. Every claim is checked against the database's own output.

In [36]:
# DESCRIBE
run("""
DESCRIBE sakila.payment
""")

,Field,Type,Null,Key,Default,Extra
0,payment_id,smallint unsigned,NO,PRI,NaN,auto_increment
1,customer_id,smallint unsigned,NO,MUL,NaN,
2,staff_id,tinyint unsigned,NO,MUL,NaN,
3,rental_id,int,YES,MUL,NaN,
4,amount,"decimal(5,2)",NO,,NaN,
5,payment_date,datetime,NO,,NaN,
6,last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [37]:
run("""
DESCRIBE sakila.rental
""")

,Field,Type,Null,Key,Default,Extra
0,rental_id,int,NO,PRI,NaN,auto_increment
1,rental_date,datetime,NO,MUL,NaN,
2,inventory_id,mediumint unsigned,NO,MUL,NaN,
3,customer_id,smallint unsigned,NO,MUL,NaN,
4,return_date,datetime,YES,,NaN,
5,staff_id,tinyint unsigned,NO,MUL,NaN,
6,last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [38]:
run("""
DESCRIBE sakila.film
""")

,Field,Type,Null,Key,Default,Extra
0,film_id,smallint unsigned,NO,PRI,NaN,auto_increment
1,title,varchar(128),NO,MUL,NaN,
2,description,text,YES,,NaN,
3,release_year,year,YES,,NaN,
4,language_id,tinyint unsigned,NO,MUL,NaN,
5,original_language_id,tinyint unsigned,YES,MUL,NaN,
6,rental_duration,tinyint unsigned,NO,,3,
7,rental_rate,"decimal(4,2)",NO,,4.99,
8,length,smallint unsigned,YES,,NaN,
9,replacement_cost,"decimal(5,2)",NO,,19.99,


In [39]:
run("""
SHOW CREATE TABLE sakila.payment
""")

,Table,Create Table
0,payment,CREATE TABLE `payment` (\n `payment_id` small...


In [40]:
run("""
SHOW CREATE TABLE sakila.rental
""")

,Table,Create Table
0,rental,CREATE TABLE `rental` (\n `rental_id` int NOT...


In [41]:
run("""
SHOW CREATE TABLE sakila.film;
""")

,Table,Create Table
0,film,CREATE TABLE `film` (\n `film_id` smallint un...



## DESCRIBE: prompt and Claude's answer
Below are the outputs of DESCRIBE sakila.payment, sakila.rental, and sakila.film

sakila.payment

```
Field    Type    Null    Key    Default    Extra
0    payment_id    smallint unsigned    NO    PRI    NaN    auto_increment
1    customer_id    smallint unsigned    NO    MUL    NaN
2    staff_id    tinyint unsigned    NO    MUL    NaN
3    rental_id    int    YES    MUL    NaN
4    amount    decimal(5,2)    NO        NaN
5    payment_date    datetime    NO        NaN
6    last_update    timestamp    YES        CURRENT_TIMESTAMP    DEFAULT_GENERATED on update CURRENT_TIMESTAMP
```

sakila.rental

```
Field    Type    Null    Key    Default    Extra
0    rental_id    int    NO    PRI    NaN    auto_increment
1    rental_date    datetime    NO    MUL    NaN
2    inventory_id    mediumint unsigned    NO    MUL    NaN
3    customer_id    smallint unsigned    NO    MUL    NaN
4    return_date    datetime    YES        NaN
5    staff_id    tinyint unsigned    NO    MUL    NaN
6    last_update    timestamp    NO        CURRENT_TIMESTAMP    DEFAULT_GENERATED on update CURRENT_TIMESTAMP
```

sakila.film

```
Field    Type    Null    Key    Default    Extra
0    film_id    smallint unsigned    NO    PRI    NaN    auto_increment
1    title    varchar(128)    NO    MUL    NaN
2    description    text    YES        NaN
3    release_year    year    YES        NaN
4    language_id    tinyint unsigned    NO    MUL    NaN
5    original_language_id    tinyint unsigned    YES    MUL    NaN
6    rental_duration    tinyint unsigned    NO        3
7    rental_rate    decimal(4,2)    NO        4.99
8    length    smallint unsigned    YES        NaN
9    replacement_cost    decimal(5,2)    NO        19.99
10    rating    enum('G','PG','PG-13','R','NC-17')    YES        G
11    special_features    set('Trailers','Commentaries','Deleted Scenes'...    YES        NaN
12    last_update    timestamp    NO        CURRENT_TIMESTAMP    DEFAULT_GENERATED on update CURRENT_TIMESTAMP
```

For each of the tables, what are the primary key, every foreign key and the table it points to, which columns allow NULL, and the data type of each money and date column?

# Response

DESCRIBE shows the `Key` column (PRI/MUL) but not which table a foreign key points to. I checked the Sakila schema file on disk (`sakila-db/sakila-schema.sql`).

The primary keys, NULL columns and data types come from your DESCRIBE output. The foreign-key targets come from the `CONSTRAINT ... FOREIGN KEY` lines in `sakila-db/sakila-schema.sql`, because DESCRIBE only marks those columns `MUL`.

## payment
- **Primary key:** `payment_id` (smallint unsigned, auto_increment)
- **Foreign keys:**
  - `customer_id` → `customer.customer_id`
  - `staff_id` → `staff.staff_id`
  - `rental_id` → `rental.rental_id` (ON DELETE SET NULL)
- **Allows NULL:**
  - `rental_id`
  - `last_update`. DESCRIBE shows it as nullable, but it defaults to CURRENT_TIMESTAMP.
- **Money column:** `amount` is `decimal(5,2)`, NOT NULL.
- **Date columns:**
  - `payment_date` is `datetime`, NOT NULL.
  - `last_update` is `timestamp`.

## rental
- **Primary key:** `rental_id` (int, auto_increment)
- **Foreign keys:**
  - `inventory_id` → `inventory.inventory_id`
  - `customer_id` → `customer.customer_id`
  - `staff_id` → `staff.staff_id`
- **Allows NULL:** `return_date`. A NULL means the rental hasn't been returned yet.
- **Money columns:** none.
- **Date columns:**
  - `rental_date` is `datetime`, NOT NULL.
  - `return_date` is `datetime`, nullable.
  - `last_update` is `timestamp`, NOT NULL.

## film
- **Primary key:** `film_id` (smallint unsigned, auto_increment)
- **Foreign keys:**
  - `language_id` → `language.language_id`
  - `original_language_id` → `language.language_id`. This is a second foreign key to the same table, and it is nullable.
- **Allows NULL:**
  - `description`
  - `release_year`
  - `original_language_id`
  - `length`
  - `rating`
  - `special_features`
- **Money columns:**
  - `rental_rate` is `decimal(4,2)`, NOT NULL, default 4.99.
  - `replacement_cost` is `decimal(5,2)`, NOT NULL, default 19.99.
- **Date columns:**
  - `release_year` is `year`, nullable.
  - `last_update` is `timestamp`, NOT NULL.

## Things to watch
- **Type mismatch:** `payment.rental_id` is `int`, but `payment.customer_id` is `smallint unsigned`. This is how the schema is defined, and it doesn't cause any problem.
- **Join to film:** `film` has no direct link to `payment` or `rental`. The path is `rental` → `inventory` (via `inventory_id`) → `film` (via `film_id`). `inventory` wasn't in your DESCRIBE output, so I haven't shown its columns here.
- **Joins and NULLs:** `payment.rental_id` can be NULL, so an inner join from `payment` to `rental` drops any such rows. In `rental`, only `return_date` can be NULL. For rental duration, filter out NULL return dates or use a LEFT JOIN.
- **Decimal types:** `amount` is DECIMAL, so SUM stays exact. That's why `ROUND(SUM(amount),2)` should match the verified 67406.56.


## EXPLAIN
EXPLAIN for Q4, Q17 and Q26: the same queries as above, with `EXPLAIN` in front, to show the plan MySQL uses to run each one.

In [42]:
# Q4 EXPLAIN
run("""
EXPLAIN
SELECT title, rental_duration
FROM sakila.film
WHERE title LIKE '%ZOO%'
  AND rental_duration >= 4
""")

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,film,None,ALL,None,None,None,None,1000,3.7,Using where


In [43]:
# Q17 EXPLAIN
run("""
EXPLAIN
SELECT rating, COUNT(*)
FROM sakila.film
WHERE rating IN ('G', 'NC-17', 'PG-13', 'PG', 'R')
GROUP BY rating
""")

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,film,None,ALL,None,None,None,None,1000,50.0,Using where; Using temporary


In [44]:
# Q26 EXPLAIN
run("""
EXPLAIN
SELECT c.customer_id, c.first_name, c.last_name
FROM sakila.customer c
JOIN sakila.rental r ON r.customer_id = c.customer_id
JOIN sakila.inventory i ON i.inventory_id = r.inventory_id
JOIN sakila.film_category fc ON fc.film_id = i.film_id
JOIN sakila.category cat ON cat.category_id = fc.category_id
WHERE cat.name = 'Horror'
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(*) > 3
""")

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,cat,None,ALL,PRIMARY,NaN,NaN,NaN,16,10.0,Using where; Using temporary
1,1,SIMPLE,fc,None,ref,"PRIMARY,fk_film_category_category",fk_film_category_category,1,sakila.cat.category_id,62,100.0,Using index
2,1,SIMPLE,i,None,ref,"PRIMARY,idx_fk_film_id",idx_fk_film_id,2,sakila.fc.film_id,4,100.0,Using index
3,1,SIMPLE,r,None,ref,"idx_fk_inventory_id,idx_fk_customer_id",idx_fk_inventory_id,3,sakila.i.inventory_id,3,100.0,NaN
4,1,SIMPLE,c,None,eq_ref,PRIMARY,PRIMARY,2,sakila.r.customer_id,1,100.0,NaN


## EXPLAIN: Claude's reading of the plans

The plans are read from the EXPLAIN output you pasted. The reasons come from the `DESCRIBE` output and the schema in `sakila-db/sakila-schema.sql`. EXPLAIN `rows` is the optimizer's estimate, not a real count. For a join, it is the estimated rows per lookup into that table.

## Q4: films with ZOO in the title and rental_duration >= 4

| Row | Table | Access type | Key used | Est. rows |
|-----|-------|-------------|----------|-----------|
| 1 | `film` | `ALL` | none (`possible_keys` is NULL) | 1000 |

- **Read in full:** `film`, all 1000 rows.
- **Why:**
  - `title LIKE '%ZOO%'` starts with a wildcard. A B-tree index on `title` is sorted from the first character, so it can't find rows that merely contain ZOO. That is why `possible_keys` is NULL even though `title` has an index.
  - `rental_duration` has no index at all.
  - With no usable index, MySQL reads every row and applies both conditions as filters (`Using where`).
- **Filtered 3.7%:** the optimizer expects about 37 of the 1000 rows to pass the filters. It is a guess.

## Q17: film count per rating

| Row | Table | Access type | Key used | Est. rows |
|-----|-------|-------------|----------|-----------|
| 1 | `film` | `ALL` | none (`possible_keys` is NULL) | 1000 |

- **Read in full:** `film`, all 1000 rows.
- **Why:**
  - `rating` has no index. The `Key` column was blank in the `DESCRIBE` output.
  - The `IN (...)` list names all five enum values, so it excludes almost nothing.
  - Even an index here would not save much, since nearly every row has to be read anyway.
- **Filtered 50.0%:** this is a default guess for an `IN` list. It is not based on the data.
- **Using temporary:** the rows aren't read in `rating` order, so MySQL builds a temporary table to hold one counter per rating for the `GROUP BY`.

## Q26: customers who rented more than 3 Horror films

| Row | Table | Access type | Key used | Est. rows |
|-----|-------|-------------|----------|-----------|
| 1 | `cat` (`category`) | `ALL` | none (`PRIMARY` was possible but not used) | 16 |
| 2 | `fc` (`film_category`) | `ref` | `fk_film_category_category` | 62 |
| 3 | `i` (`inventory`) | `ref` | `idx_fk_film_id` | 4 |
| 4 | `r` (`rental`) | `ref` | `idx_fk_inventory_id` | 3 |
| 5 | `c` (`customer`) | `eq_ref` | `PRIMARY` | 1 |

The rows are listed in join order. MySQL starts at `cat`, then looks up each next table using values from the previous row.

- **Read in full:** `cat` (`category`), all 16 rows. It is the only full scan in this plan.
- **Why:**
  - The filter is `cat.name = 'Horror'`, and `category.name` has no index. The only index is the `PRIMARY` key on `category_id`, which the query doesn't filter on.
  - The table has only 16 rows, so scanning it is cheap.
  - Filtered 10.0% means the optimizer expects about 1 or 2 matching rows. In reality, Horror is one category.
- **Why `cat` goes first:** it has the most selective filter, so it gives the smallest starting set. Every join after it is an index lookup.
- **Row 2, `fc`:** `ref` on `fk_film_category_category`. `key_len` 1 is a tinyint unsigned (`category_id`). About 62 films are expected per category. `Using index` means the index alone answers the query, with no table row read.
- **Row 3, `i`:** `ref` on `idx_fk_film_id`. `key_len` 2 is a smallint unsigned (`film_id`). About 4 inventory copies are expected per film. It is also `Using index` (covering).
- **Row 4, `r`:** `ref` on `idx_fk_inventory_id`. `key_len` 3 is a mediumint unsigned (`inventory_id`). About 3 rentals are expected per inventory copy. `Extra` is NULL, so MySQL must read the `rental` table row. `idx_fk_customer_id` was also possible, but it isn't the join path being followed here.
- **Row 5, `c`:** `eq_ref` on `PRIMARY` with `key_len` 2 (`customer_id`). Exactly one customer row is expected per rental. This is the best join access type.
- **Using temporary (row 1):** this applies to the whole query. The `GROUP BY` is on customer columns, but the rows arrive in category order, so MySQL uses a temporary table to group them before applying `HAVING COUNT(*) > 3`.
- **Estimated work:** about 1.6 category rows × 62 × 4 × 3 × 1 is roughly 1,200 `rental` lookups. These are index lookups, not a scan of the 16,044 rental rows.

## Summary

| Query | Table read in full | Rows | Why |
|-------|--------------------|------|-----|
| Q4 | `film` | 1000 | A leading-wildcard `LIKE` can't use the `title` index, and `rental_duration` has no index. |
| Q17 | `film` | 1000 | `rating` has no index, and the `IN` list matches nearly every row. |
| Q26 | `category` | 16 | `category.name` has no index, and the table is tiny. |

None of the plans use access type `index` or `range`. Q26 does show `Using index` on `fc` and `i`. That is a different thing: it means a covering index, not a full index scan.


## EXPLAIN: which column confirms each claim

Most of my statements are confirmed by the EXPLAIN output. Three are not confirmed, and one has wrong wording.

Confirmed by EXPLAIN

| Statement | Column | Value that confirms it |
|---|---|---|
| Q4 reads `film` in full | `type`, `key`, `rows` | `ALL`, NULL, 1000 |
| Q17 reads `film` in full | `type`, `key`, `rows` | `ALL`, NULL, 1000 |
| Q26 reads `category` in full | `type`, `key`, `rows` | `ALL`, NULL, 16 |
| Q4 and Q17 have no usable index | `possible_keys` | NULL in both |
| Q4 applies both conditions as filters | `Extra` | `Using where` |
| Q4 expects about 37 rows to pass | `rows` × `filtered` | 1000 × 3.7% = 37 |
| Q17 builds a temporary table for the `GROUP BY` | `Extra` | `Using temporary` |
| `PRIMARY` was possible on `cat` but not used | `possible_keys` vs `key` | `PRIMARY` vs NULL |
| `cat` is the first table in the join | row order | `cat` is first |
| `fc` is an index lookup | `type`, `key`, `ref` | `ref`, `fk_film_category_category`, `cat.category_id` |
| `fc` and `i` are covering indexes | `Extra` | `Using index` on both |
| `r` reads the table row | `Extra` | NULL, meaning no `Using index` |
| `idx_fk_customer_id` was possible for `r` but not used | `possible_keys` vs `key` | listed vs `idx_fk_inventory_id` |
| `c` gets exactly one row per rental | `type`, `key`, `rows` | `eq_ref`, `PRIMARY`, 1 |
| `rows` is per lookup, not a total | `rows` on `r` | 3, far below the table's size |
| No plan uses `index` or `range` | `type` | only `ALL`, `ref`, `eq_ref` |
| `Using index` is not access type `index` | `Extra` vs `type` | `fc` and `i` show `Using index` while their `type` is `ref` |

`key_len` values 1, 2, 3 and 2 fit the column types I named: tinyint, smallint, mediumint and smallint. That is consistent with them, not proof. The `rental` and `film` types come from your DESCRIBE output. I took the `category_id` type from my knowledge of the schema and didn't check it.

Not confirmed by EXPLAIN
- **Why a leading-wildcard `LIKE` can't use the index:** EXPLAIN only shows `possible_keys` NULL. It doesn't say why. The reason is how B-tree indexes work.
- **Why `cat` goes first and why `Using temporary` appears:** EXPLAIN shows the join order and the temp table. The reasons I gave (most selective filter, rows arriving in category order) are my inference.
- **"`filtered` 50.0% is a default guess" (Q17) and the 10.0% on `cat`:** I stated these as fact, but EXPLAIN doesn't say how it calculated them, and I haven't verified it. Treat them as unverified.
- **Facts from outside EXPLAIN:**
  - Horror is one category.
  - The `IN` list covers every rating.
  - The `rental` table has 16,044 rows.

Contradiction and a wording error
- **Q17 `filtered` 50.0 vs "excludes almost nothing":** the optimizer estimates half the rows pass. My statement that nearly every row passes comes from the enum definition, not from EXPLAIN. The two disagree, and it is only the estimate that is off.
- **"Roughly 1,200 `rental` lookups" is wrong:** the figures are about 1.6 × 62 × 4 ≈ 400 lookups into `rental`. Each lookup returns about 3 rows, so about 1,200 rows are read. This wording is in `docs/query_plans.md` under Q26 and is left there as the record.
- **"eq_ref is the best join access type":** this is slightly overstated. `const` and `system` rank above it, but those only apply to single-row tables.



## EXPLAIN: an index that could change a full scan

**Plan:** Q17, a full scan (`type` ALL) of `film`, the largest table scanned in these plans (1000 rows, versus 16 for `category` in Q26). Q4 also scans `film` in full, but no ordinary index can help it, because `LIKE '%ZOO%'` starts with a wildcard.

**Index that could change it** (proposed only, not created):

```sql
CREATE INDEX idx_film_rating ON sakila.film (rating);
```

`rating` is the only column Q17 filters and groups on, so this index alone would cover the query. MySQL could read the narrow index instead of the whole table and could group in index order, which would remove `Using temporary`.

**Cost on INSERT:** `film` currently has the primary key plus three secondary indexes (`idx_title`, `idx_fk_language_id`, `idx_fk_original_language_id`). Every `INSERT` would also have to add an entry to the new index (a fourth secondary index), and every `UPDATE` of `rating` and every `DELETE` would have to maintain it. It also takes extra storage. Each entry is small, since `rating` is a 1-byte enum plus the primary key, but the index has only 5 distinct values.

**Would it help this query, judged from the `rows` column?** No: `rows` is 1000 and the `IN` list names all five ratings, so an index on `rating` would still have to visit about 1000 entries, and the estimated row count would barely drop, with the only likely gain being no temporary table.

## Five simple facts: Claude's answers first

Five simple facts about Sakila:

1. **Films, actors and customers:** there are 1,000 films, 200 actors and 599 customers.
2. **Rentals:** there are 16,044 rentals (and 16,044 payments).
3. **Most common rating:** PG-13, with 223 of the 1,000 films. The others are NC-17 (210), R (195), PG (194) and G (178).
4. **Payment dates:** the earliest `payment_date` is 2005-05-24 22:53:30 and the latest is 2006-02-14 15:16:03.
5. **Categories:** there are 16 film categories.

These numbers were counted from the rows in `sakila-db/sakila-data.sql`, not by querying the Cloud SQL database. The counts match the verified Sakila v1.5 numbers (1,000 films, 200 actors, 599 customers, 16,044 rentals), but the rating counts, date range and category count have not been checked with SQL in this notebook.

## Five simple facts: one query per fact. Expected results, so each one can be checked against the output:

1. films 1000, actors 200, customers 599
2. rentals 16044, payments 16044
3. PG-13 223, NC-17 210, R 195, PG 194, G 178
4. earliest 2005-05-24 22:53:30, latest 2006-02-14 15:16:03
5. 16 categories

In [45]:
# Fact 1: films, actors and customers
run("""
SELECT
  (SELECT COUNT(*) FROM sakila.film)     AS films,
  (SELECT COUNT(*) FROM sakila.actor)    AS actors,
  (SELECT COUNT(*) FROM sakila.customer) AS customers
""")

,films,actors,customers
0,1000,200,599


In [46]:
# Fact 2: rentals and payments
run("""
SELECT
  (SELECT COUNT(*) FROM sakila.rental)  AS rentals,
  (SELECT COUNT(*) FROM sakila.payment) AS payments
""")

,rentals,payments
0,16044,16044


In [47]:
# Fact 3: films per rating, most common first
run("""
SELECT rating, COUNT(*) AS films
FROM sakila.film
GROUP BY rating
ORDER BY films DESC, rating
""")

,rating,films
0,PG-13,223
1,NC-17,210
2,R,195
3,PG,194
4,G,178


In [48]:
# Fact 4: earliest and latest payment_date
run("""
SELECT MIN(payment_date) AS earliest, MAX(payment_date) AS latest
FROM sakila.payment
""")

,earliest,latest
0,2005-05-24 22:53:30,2006-02-14 15:16:03


In [49]:
# Fact 5: number of categories
run("""
SELECT COUNT(*) AS categories
FROM sakila.category
""")

,categories
0,16


## Five simple facts: the check

| # | Fact | Claude's answer | Query | Actual result | Match (Y/N) |
|---|------|-----------------|-------|---------------|-------------|
| 1 | Films, actors and customers | 1,000 films, 200 actors, 599 customers | `SELECT (SELECT COUNT(*) FROM sakila.film) AS films, (SELECT COUNT(*) FROM sakila.actor) AS actors, (SELECT COUNT(*) FROM sakila.customer) AS customers` | films 1000, actors 200, customers 599 | Y |
| 2 | Rentals | 16,044 rentals (and 16,044 payments) | `SELECT (SELECT COUNT(*) FROM sakila.rental) AS rentals, (SELECT COUNT(*) FROM sakila.payment) AS payments` | rentals 16044, payments 16044 | Y |
| 3 | Most common rating | PG-13 with 223 films (NC-17 210, R 195, PG 194, G 178) | `SELECT rating, COUNT(*) AS films FROM sakila.film GROUP BY rating ORDER BY films DESC, rating` | PG-13 223, NC-17 210, R 195, PG 194, G 178 | Y |
| 4 | Earliest and latest payment_date | earliest 2005-05-24 22:53:30, latest 2006-02-14 15:16:03 | `SELECT MIN(payment_date) AS earliest, MAX(payment_date) AS latest FROM sakila.payment` | earliest 2005-05-24 22:53:30, latest 2006-02-14 15:16:03 | Y |
| 5 | Number of categories | 16 | `SELECT COUNT(*) AS categories FROM sakila.category` | 16 | Y |



## A query written by Claude, checked independently: which store has more customers?

Claude got all five facts right, so this uses the fallback question: which store has more customers?

**Query:** count customers per store and list the larger store first.

In [50]:
# Which store has more customers
run("""
SELECT store_id, COUNT(*) AS customers
FROM sakila.customer
GROUP BY store_id
ORDER BY customers DESC
""")

,store_id,customers
0,1,326
1,2,273


**Independent check:** the total from a plain `COUNT(*)` on `sakila.customer` must equal the sum of the per-store counts from the `GROUP BY`.

In [51]:
# Check: COUNT(*) must equal the sum of the per-store counts
run("""
SELECT
  (SELECT COUNT(*) FROM sakila.customer) AS total_customers,
  SUM(g.customers)                       AS sum_of_store_counts
FROM (
  SELECT store_id, COUNT(*) AS customers
  FROM sakila.customer
  GROUP BY store_id
) g
""")

,total_customers,sum_of_store_counts
0,599,599


**Do the two numbers agree?** Expected, from counting the rows in `sakila-db/sakila-data.sql` (confirm against the two outputs above): store 1 has 326 customers and store 2 has 273, so store 1 has more. The check should show 599 for both `total_customers` and `sum_of_store_counts` (326 + 273 = 599, the verified customer count). If the two numbers differ, some customers would have a NULL or unexpected `store_id`.

**What Claude was reliable on, and what needed checking:** Claude was reliable on the five Sakila facts, which all matched the query results, and on reading the EXPLAIN plans, where the access types, keys and row estimates were confirmed by the output. It needed checking wherever it went past the data: the "1,200 rental lookups" figure (it should have been about 400 lookups returning about 1,200 rows), the overstated "eq_ref is best" claim, the unverified claims about how `filtered` is calculated, and the store-count answer, which is only an expectation until you run those queries.